# 07 — Testy jednostkowe warstw zespolonych

Weryfikacja poprawności samodzielnie zaimplementowanych warstw. Testy operują na
tensorach losowych i nie wymagają zbioru danych ani akceleratora graficznego.

Test przeuczenia pojedynczej partii, wykonywany przed każdym eksperymentem,
potwierdza jedynie drożność ścieżki gradientowej. Nie stanowi dowodu poprawności
matematycznej, ponieważ błędnie zaimplementowana, lecz różniczkowalna warstwa
również mogłaby go przejść.

**Wynik:** 39 z 39 testów zakończonych powodzeniem

## 1. Konfiguracja

In [1]:
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- parametry architektury, zgodne z notatnikiem 05 ---
N_FFT, WIN_LENGTH, HOP_LENGTH = 512, 400, 160
SR, SEG_SEC = 16000, 1.0
SEG_LEN = int(SEG_SEC * SR)
N_FRAMES = SEG_LEN // HOP_LENGTH + 1      # 101
N_BINS = N_FFT // 2 + 1                   # 257

N_CLASSES = 6
CH = (23, 45, 90, 90)
RNN_HIDDEN = 90

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print("urządzenie:", DEVICE)
print("Testy operują na tensorach losowych i nie wymagają zbioru danych.")

urządzenie: cpu
Testy operują na tensorach losowych i nie wymagają zbioru danych.


In [2]:
torch.manual_seed(0)
np.random.seed(0)
DEV = DEVICE
TOL = 1e-5         

wyniki = []

def sprawdz(nazwa, warunek, szczegol=""):
    wyniki.append((nazwa, bool(warunek)))
    status = "OK   " if warunek else "BLAD "
    print(f"[{status}] {nazwa}" + (f"   {szczegol}" if szczegol else ""))
    return bool(warunek)

## 2. Definicje warstw

Definicje identyczne z notatnikami 05 i 06, powtórzone dla samodzielności
notatnika.

In [3]:
class ComplexConv2d(nn.Module):
    def __init__(self, cin, cout, k=3, padding=1):
        super().__init__()
        self.cr = nn.Conv2d(cin, cout, k, padding=padding, bias=False)
        self.ci = nn.Conv2d(cin, cout, k, padding=padding, bias=False)
        self._init_complex(cin * k * k)

    def _init_complex(self, fan_in):
        sigma = 1.0 / math.sqrt(2 * fan_in)
        for w_r, w_i in [(self.cr.weight, self.ci.weight)]:
            with torch.no_grad():
                mag = torch.from_numpy(
                    np.random.rayleigh(sigma, size=tuple(w_r.shape))).float()
                phase = torch.empty_like(w_r).uniform_(-math.pi, math.pi)
                w_r.copy_(mag * torch.cos(phase))
                w_i.copy_(mag * torch.sin(phase))

    def forward(self, z):
        zr, zi = z.real, z.imag
        return torch.complex(self.cr(zr) - self.ci(zi),
                             self.cr(zi) + self.ci(zr))

In [4]:
class ComplexBatchNorm2d(nn.Module):
    def __init__(self, c, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps, self.momentum = eps, momentum
        self.g_rr = nn.Parameter(torch.full((c,), 1 / math.sqrt(2)))
        self.g_ii = nn.Parameter(torch.full((c,), 1 / math.sqrt(2)))
        self.g_ri = nn.Parameter(torch.zeros(c))
        self.b_r = nn.Parameter(torch.zeros(c))
        self.b_i = nn.Parameter(torch.zeros(c))
        for n, v in [("rm_r", torch.zeros(c)), ("rm_i", torch.zeros(c)),
                     ("rv_rr", torch.full((c,), 1 / math.sqrt(2))),
                     ("rv_ii", torch.full((c,), 1 / math.sqrt(2))),
                     ("rv_ri", torch.zeros(c))]:
            self.register_buffer(n, v)

    def forward(self, z):
        zr, zi = z.real, z.imag
        dims = (0, 2, 3)
        if self.training:
            mr, mi = zr.mean(dims), zi.mean(dims)
            with torch.no_grad():
                self.rm_r.mul_(1 - self.momentum).add_(self.momentum * mr)
                self.rm_i.mul_(1 - self.momentum).add_(self.momentum * mi)
        else:
            mr, mi = self.rm_r, self.rm_i

        v = lambda t: t.view(1, -1, 1, 1)
        xr, xi = zr - v(mr), zi - v(mi)

        if self.training:
            vrr = (xr * xr).mean(dims) + self.eps
            vii = (xi * xi).mean(dims) + self.eps
            vri = (xr * xi).mean(dims)
            with torch.no_grad():
                self.rv_rr.mul_(1 - self.momentum).add_(self.momentum * vrr)
                self.rv_ii.mul_(1 - self.momentum).add_(self.momentum * vii)
                self.rv_ri.mul_(1 - self.momentum).add_(self.momentum * vri)
        else:
            vrr, vii, vri = self.rv_rr, self.rv_ii, self.rv_ri

        
        det = (vrr * vii - vri * vri).clamp_min(self.eps)
        s = torch.sqrt(det)
        t = torch.sqrt(vrr + vii + 2 * s).clamp_min(self.eps)
        inv = 1.0 / (s * t)
        wrr, wii, wri = (vii + s) * inv, (vrr + s) * inv, -vri * inv

        nr = v(wrr) * xr + v(wri) * xi
        ni = v(wri) * xr + v(wii) * xi

        outr = v(self.g_rr) * nr + v(self.g_ri) * ni + v(self.b_r)
        outi = v(self.g_ri) * nr + v(self.g_ii) * ni + v(self.b_i)
        return torch.complex(outr, outi)


def crelu(z):
    return torch.complex(F.relu(z.real), F.relu(z.imag))

In [5]:
class ComplexLinear(nn.Module):
    def __init__(self, cin, cout, bias=True):
        super().__init__()
        self.lr = nn.Linear(cin, cout, bias=False)
        self.li = nn.Linear(cin, cout, bias=False)
        self.use_bias = bias
        if bias:
            self.b_r = nn.Parameter(torch.zeros(cout))
            self.b_i = nn.Parameter(torch.zeros(cout))
        sigma = 1.0 / math.sqrt(2 * cin)
        with torch.no_grad():
            mag = torch.from_numpy(
                np.random.rayleigh(sigma, size=tuple(self.lr.weight.shape))).float()
            phase = torch.empty_like(self.lr.weight).uniform_(-math.pi, math.pi)
            self.lr.weight.copy_(mag * torch.cos(phase))
            self.li.weight.copy_(mag * torch.sin(phase))

    def forward(self, z):
        zr, zi = z.real, z.imag
        out_r = self.lr(zr) - self.li(zi)
        out_i = self.lr(zi) + self.li(zr)
        if self.use_bias:
            out_r = out_r + self.b_r
            out_i = out_i + self.b_i
        return torch.complex(out_r, out_i)


class ComplexGRUCell(nn.Module):
    def __init__(self, cin, hidden):
        super().__init__()
        self.hidden = hidden
        self.x2h = ComplexLinear(cin, 3 * hidden)
        self.h2h = ComplexLinear(hidden, 3 * hidden, bias=False)

    def forward(self, x, h):
        gx = self.x2h(x)
        gh = self.h2h(h)
        H = self.hidden
        z = torch.sigmoid((gx[:, :H] + gh[:, :H]).abs())          
        r = torch.sigmoid((gx[:, H:2*H] + gh[:, H:2*H]).abs())   
        n = gx[:, 2*H:] + r.to(gh.dtype) * gh[:, 2*H:]
        n = torch.complex(torch.tanh(n.real), torch.tanh(n.imag)) 
        zc = z.to(h.dtype)
        return (1 - zc) * n + zc * h


class ComplexBiGRU(nn.Module):
    def __init__(self, cin, hidden):
        super().__init__()
        self.fwd = ComplexGRUCell(cin, hidden)
        self.bwd = ComplexGRUCell(cin, hidden)
        self.hidden = hidden

    def forward(self, x):                     
        B, T, _ = x.shape
        h = torch.zeros(B, self.hidden, dtype=x.dtype, device=x.device)
        out_f = []
        for t in range(T):
            h = self.fwd(x[:, t], h)
            out_f.append(h)
        h = torch.zeros(B, self.hidden, dtype=x.dtype, device=x.device)
        out_b = []
        for t in reversed(range(T)):
            h = self.bwd(x[:, t], h)
            out_b.append(h)
        out_b.reverse()
        return torch.cat([torch.stack(out_f, 1), torch.stack(out_b, 1)], dim=-1)

In [6]:
lay = ComplexLinear(8, 5).to(DEV)
with torch.no_grad():
    lay.b_r.normal_(); lay.b_i.normal_()
z = torch.complex(torch.randn(4, 8), torch.randn(4, 8)).to(DEV)
W = torch.complex(lay.lr.weight, lay.li.weight)
b = torch.complex(lay.b_r, lay.b_i)
sprawdz("warstwa liniowa: zgodnosc z y = Wz + b (bias != 0)",
        torch.allclose(lay(z), z @ W.T + b, atol=TOL),
        f"maks. roznica {(lay(z) - (z @ W.T + b)).abs().max():.2e}")

[OK   ] warstwa liniowa: zgodnosc z y = Wz + b (bias != 0)   maks. roznica 2.46e-07


True

In [7]:
class ComplexConvBlock(nn.Module):
    def __init__(self, cin, cout, pool):
        super().__init__()
        self.conv = ComplexConv2d(cin, cout)
        self.bn = ComplexBatchNorm2d(cout)
        self.pool = pool

    def forward(self, z):
        z = crelu(self.bn(self.conv(z)))
        pf, pt = self.pool
        mag = z.abs()
        _, idx = F.max_pool2d(mag, (pf, pt), return_indices=True)
        B, C, H, W = z.shape
        flat_r = z.real.reshape(B, C, -1).gather(2, idx.reshape(B, C, -1))
        flat_i = z.imag.reshape(B, C, -1).gather(2, idx.reshape(B, C, -1))
        shp = idx.shape
        return torch.complex(flat_r.reshape(shp), flat_i.reshape(shp))


class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w = nn.Linear(dim, 1)

    def forward(self, x):
        a = torch.softmax(self.w(x), dim=1)
        return (x * a).sum(dim=1)


class CVRCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES, ch=CH, rnn_hidden=RNN_HIDDEN):
        super().__init__()
        pools = [(4, 2), (4, 2), (4, 1), (2, 1)]
        blocks, cin = [], 1
        for cout, p in zip(ch, pools):
            blocks.append(ComplexConvBlock(cin, cout, p)); cin = cout
        self.blocks = nn.ModuleList(blocks)

        with torch.no_grad():
            d = torch.complex(torch.zeros(1, 1, N_BINS, N_FRAMES),
                              torch.zeros(1, 1, N_BINS, N_FRAMES))
            for b in self.blocks:
                d = b(d)
            c, f, t = d.shape[1:]
        self.seq_dim, self.seq_len = c * f, t

        self.rnn = ComplexBiGRU(self.seq_dim, rnn_hidden)
        self.att = AttentionPool(2 * rnn_hidden)
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2 * rnn_hidden, 64), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, z):
        for b in self.blocks:
            z = b(z)
        z = z.permute(0, 3, 1, 2).flatten(2)     
        z = self.rnn(z)                         
        return self.head(self.att(z.abs()))      


m = CVRCNN().to(DEVICE)
n_par = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"sekwencja: dlugosc {m.seq_len}, wymiar {m.seq_dim} zespolonych")
print(f"parametry uczone (rzeczywiste): {n_par:,}")


del m

sekwencja: dlugosc 25, wymiar 180 zespolonych
parametry uczone (rzeczywiste): 543,819


## 3. Zespolona warstwa splotowa

Porównanie z bezpośrednim obliczeniem na częściach rzeczywistej i urojonej,
zgodnie z rozwinięciem mnożenia zespolonego. Test liniowości stanowi niezależne
potwierdzenie, ponieważ weryfikuje własność matematyczną, nie zaś zgodność z drugą
implementacją tego samego wzoru.

In [8]:
conv = ComplexConv2d(3, 5, k=3, padding=1).to(DEV)
z = torch.complex(torch.randn(4, 3, 16, 12), torch.randn(4, 3, 16, 12)).to(DEV)

out = conv(z)

Wr, Wi = conv.cr.weight, conv.ci.weight
ref_r = F.conv2d(z.real, Wr, padding=1) - F.conv2d(z.imag, Wi, padding=1)
ref_i = F.conv2d(z.imag, Wr, padding=1) + F.conv2d(z.real, Wi, padding=1)

sprawdz("splot: czesc rzeczywista",
        torch.allclose(out.real, ref_r, atol=TOL),
        f"maks. roznica {(out.real - ref_r).abs().max():.2e}")
sprawdz("splot: czesc urojona",
        torch.allclose(out.imag, ref_i, atol=TOL),
        f"maks. roznica {(out.imag - ref_i).abs().max():.2e}")
sprawdz("splot: typ wyjscia zespolony", out.is_complex())
sprawdz("splot: ksztalt wyjscia", tuple(out.shape) == (4, 5, 16, 12),
        str(tuple(out.shape)))

[OK   ] splot: czesc rzeczywista   maks. roznica 0.00e+00
[OK   ] splot: czesc urojona   maks. roznica 0.00e+00
[OK   ] splot: typ wyjscia zespolony
[OK   ] splot: ksztalt wyjscia   (4, 5, 16, 12)


True

In [9]:
z1 = torch.complex(torch.randn(2, 3, 8, 8), torch.randn(2, 3, 8, 8)).to(DEV)
z2 = torch.complex(torch.randn(2, 3, 8, 8), torch.randn(2, 3, 8, 8)).to(DEV)
alpha = torch.complex(torch.tensor(0.7), torch.tensor(-1.3)).to(DEV)

lewa = conv(alpha * z1 + z2)
prawa = alpha * conv(z1) + conv(z2)

sprawdz("splot: liniowosc w ciele C",
        torch.allclose(lewa, prawa, atol=1e-4),
        f"maks. roznica {(lewa - prawa).abs().max():.2e}")

[OK   ] splot: liniowosc w ciele C   maks. roznica 1.51e-06


True

## 4. Inicjalizacja wag

Weryfikacja zgodności rozkładu modułu z rozkładem Rayleigha oraz jednostajnego
rozkładu kąta fazowego. Dla rozkładu Rayleigha o parametrze skali σ zachodzi
E[|W|²] = 2σ², co przy przyjętym σ = 1/√(2·fan_in) odpowiada E[|W|²] = 1/fan_in.

In [10]:
duzy = ComplexConv2d(64, 64, k=3, padding=1)
W = torch.complex(duzy.cr.weight.detach(), duzy.ci.weight.detach()).flatten()
n_in = 64 * 3 * 3
sigma = 1.0 / math.sqrt(2 * n_in)

mag, faza = W.abs(), torch.angle(W)
oczek_e2 = 1.0 / n_in

sprawdz("inicjalizacja: E[|W|^2] = 1/n_in",
        abs((mag ** 2).mean().item() - oczek_e2) / oczek_e2 < 0.06,
        f"zmierzone {(mag**2).mean():.3e}, oczekiwane {oczek_e2:.3e}")

sprawdz("inicjalizacja: srednia modulu zgodna z rozkladem Rayleigha",
        abs(mag.mean().item() - sigma * math.sqrt(math.pi / 2)) / (sigma * math.sqrt(math.pi / 2)) < 0.06,
        f"zmierzone {mag.mean():.3e}, oczekiwane {sigma*math.sqrt(math.pi/2):.3e}")

sprawdz("inicjalizacja: faza o zerowej wartosci oczekiwanej",
        abs(faza.mean().item()) < 0.05,
        f"srednia fazy {faza.mean():.4f}")
sprawdz("inicjalizacja: faza pokrywa pelny przedzial",
        faza.min() < -3.0 and faza.max() > 3.0,
        f"zakres [{faza.min():.2f}, {faza.max():.2f}]")

[OK   ] inicjalizacja: E[|W|^2] = 1/n_in   zmierzone 1.726e-03, oczekiwane 1.736e-03
[OK   ] inicjalizacja: srednia modulu zgodna z rozkladem Rayleigha   zmierzone 3.678e-02, oczekiwane 3.693e-02
[OK   ] inicjalizacja: faza o zerowej wartosci oczekiwanej   srednia fazy -0.0011
[OK   ] inicjalizacja: faza pokrywa pelny przedzial   zakres [-3.14, 3.14]


True

## 5. Zespolona normalizacja wsadowa

Sprawdzenie sprowadzenia wartości oczekiwanej do zera oraz wybielenia danych.
Przy początkowych wartościach parametrów uczonych wariancja każdej ze składowych
powinna wynosić ½, a ich kowariancja zero.

Poprawność postaci zamkniętej odwrotnego pierwiastka macierzy kowariancji
weryfikowana jest niezależnie, na zbiorze losowo generowanych macierzy.

In [11]:
cbn = ComplexBatchNorm2d(4).to(DEV).train()

baza = torch.randn(256, 4, 10, 10)
x_r = 3.0 * baza + 1.5
x_i = 1.5 * baza + 2.0 * torch.randn(256, 4, 10, 10) - 0.8
z = torch.complex(x_r, x_i).to(DEV)

with torch.no_grad():
    out = cbn(z)

r, i = out.real, out.imag
sr, si = r.mean(dim=(0, 2, 3)), i.mean(dim=(0, 2, 3))
vr = (r ** 2).mean(dim=(0, 2, 3)) - sr ** 2
vi = (i ** 2).mean(dim=(0, 2, 3)) - si ** 2
vri = (r * i).mean(dim=(0, 2, 3)) - sr * si

sprawdz("CBN: zerowa wartosc oczekiwana",
        sr.abs().max() < 1e-4 and si.abs().max() < 1e-4,
        f"maks. |srednia| {max(sr.abs().max(), si.abs().max()):.2e}")
sprawdz("CBN: wariancja skladowych rowna 1/2",
        (vr - 0.5).abs().max() < 5e-3 and (vi - 0.5).abs().max() < 5e-3,
        f"var_R {vr.mean():.4f}, var_I {vi.mean():.4f}")
sprawdz("CBN: zerowa kowariancja skladowych",
        vri.abs().max() < 5e-3,
        f"maks. |kowariancja| {vri.abs().max():.2e}")

zr0, zi0 = z.real, z.imag
kow0 = ((zr0 - zr0.mean()) * (zi0 - zi0.mean())).mean()
sprawdz("CBN: dane wejsciowe faktycznie skorelowane",
        kow0.abs() > 1.0, f"kowariancja wejscia {kow0:.3f}")

[OK   ] CBN: zerowa wartosc oczekiwana   maks. |srednia| 2.26e-07
[OK   ] CBN: wariancja skladowych rowna 1/2   var_R 0.5000, var_I 0.5000
[OK   ] CBN: zerowa kowariancja skladowych   maks. |kowariancja| 7.24e-07
[OK   ] CBN: dane wejsciowe faktycznie skorelowane   kowariancja wejscia 4.472


True

In [12]:
def inv_sqrt_2x2(vrr, vii, vri):
    det = vrr * vii - vri * vri
    s = math.sqrt(det)
    t = math.sqrt(vrr + vii + 2 * s)
    inv = 1.0 / (s * t)
    return np.array([[(vii + s) * inv, -vri * inv],
                     [-vri * inv, (vrr + s) * inv]])


rng = np.random.default_rng(7)
maks_blad = 0.0
for _ in range(200):
    A = rng.normal(size=(2, 300))
    V = np.cov(A)
    Wm = inv_sqrt_2x2(V[0, 0], V[1, 1], V[0, 1])
    maks_blad = max(maks_blad, np.abs(Wm @ V @ Wm - np.eye(2)).max())

sprawdz("odwrotny pierwiastek: V^(-1/2) V V^(-1/2) = I",
        maks_blad < 1e-8, f"maks. odchylenie {maks_blad:.2e}")

[OK   ] odwrotny pierwiastek: V^(-1/2) V V^(-1/2) = I   maks. odchylenie 7.77e-16


True

## 6. Aktywacja CReLU

In [13]:
z = torch.complex(torch.randn(64, 8), torch.randn(64, 8)).to(DEV)
out = crelu(z)

sprawdz("CReLU: czesc rzeczywista",
        torch.allclose(out.real, torch.clamp(z.real, min=0), atol=TOL))
sprawdz("CReLU: czesc urojona",
        torch.allclose(out.imag, torch.clamp(z.imag, min=0), atol=TOL))
sprawdz("CReLU: brak wartosci ujemnych",
        (out.real >= 0).all() and (out.imag >= 0).all())


zmiana = (torch.angle(out) - torch.angle(z)).abs() > 1e-6
sprawdz("CReLU: modyfikuje kat fazowy czesci argumentow",
        zmiana.any(), f"udzial zmodyfikowanych {100*zmiana.float().mean():.1f}%")

[OK   ] CReLU: czesc rzeczywista
[OK   ] CReLU: czesc urojona
[OK   ] CReLU: brak wartosci ujemnych
[OK   ] CReLU: modyfikuje kat fazowy czesci argumentow   udzial zmodyfikowanych 74.2%


True

## 7. Zespolona warstwa redukująca

Porównanie z ręcznym przejściem po oknach oraz weryfikacja zachowania kąta
fazowego wybranego elementu.

In [14]:

z = torch.complex(torch.randn(3, 2, 8, 8), torch.randn(3, 2, 8, 8)).to(DEV)

with torch.no_grad():
    mag = z.abs()
    _, idx = F.max_pool2d(mag, (2, 2), return_indices=True)
    B, C, H, W = z.shape
    fr = z.real.reshape(B, C, -1).gather(2, idx.reshape(B, C, -1)).reshape(idx.shape)
    fi = z.imag.reshape(B, C, -1).gather(2, idx.reshape(B, C, -1)).reshape(idx.shape)
    out = torch.complex(fr, fi)

ref = torch.zeros_like(out)
for b in range(B):
    for c in range(C):
        for i in range(0, H, 2):
            for j in range(0, W, 2):
                okno = z[b, c, i:i+2, j:j+2].reshape(-1)
                ref[b, c, i//2, j//2] = okno[okno.abs().argmax()]

sprawdz("redukcja: zgodnosc z selekcja wedlug modulu",
        torch.allclose(out, ref, atol=TOL),
        f"maks. roznica {(out - ref).abs().max():.2e}")
sprawdz("redukcja: wartosc wyjsciowa zespolona", out.is_complex())


zgodna_faza = torch.allclose(torch.angle(out), torch.angle(ref), atol=TOL)
sprawdz("redukcja: zachowanie kata fazowego wybranego elementu", zgodna_faza)


maxmod = F.max_pool2d(z.abs(), (2, 2))
sprawdz("redukcja: modul wyjscia rowny maksimum w oknie",
        torch.allclose(out.abs(), maxmod, atol=TOL))

[OK   ] redukcja: zgodnosc z selekcja wedlug modulu   maks. roznica 0.00e+00
[OK   ] redukcja: wartosc wyjsciowa zespolona
[OK   ] redukcja: zachowanie kata fazowego wybranego elementu
[OK   ] redukcja: modul wyjscia rowny maksimum w oknie


True

## 8. Zespolona jednostka rekurencyjna

Weryfikacja rzeczywistego charakteru bramek, ich zakresu oraz zgodności
aktualizacji stanu z równaniem definicyjnym.

In [15]:
cell = ComplexGRUCell(12, 8).to(DEV)
x = torch.complex(torch.randn(5, 12), torch.randn(5, 12)).to(DEV)
h0 = torch.complex(torch.randn(5, 8), torch.randn(5, 8)).to(DEV)

with torch.no_grad():
    h1 = cell(x, h0)

    gx, gh = cell.x2h(x), cell.h2h(h0)
    H = cell.hidden
    z_t = torch.sigmoid((gx[:, :H] + gh[:, :H]).abs())
    r_t = torch.sigmoid((gx[:, H:2*H] + gh[:, H:2*H]).abs())

sprawdz("CV-GRU: bramka aktualizacji rzeczywista",
        not z_t.is_complex())
sprawdz("CV-GRU: bramka resetu rzeczywista",
        not r_t.is_complex())
sprawdz("CV-GRU: wartosci bramek w przedziale [1/2, 1)",
        (z_t >= 0.5).all() and (z_t < 1).all() and (r_t >= 0.5).all() and (r_t < 1).all(),
        f"min {min(z_t.min(), r_t.min()):.4f}, maks {max(z_t.max(), r_t.max()):.4f}")
sprawdz("CV-GRU: stan ukryty zespolony", h1.is_complex())
sprawdz("CV-GRU: ksztalt stanu", tuple(h1.shape) == (5, 8), str(tuple(h1.shape)))

with torch.no_grad():
    n_t = gx[:, 2*H:] + r_t.to(gh.dtype) * gh[:, 2*H:]
    n_t = torch.complex(torch.tanh(n_t.real), torch.tanh(n_t.imag))
    ref = (1 - z_t.to(h0.dtype)) * n_t + z_t.to(h0.dtype) * h0

sprawdz("CV-GRU: zgodnosc z rownaniem aktualizacji stanu",
        torch.allclose(h1, ref, atol=TOL),
        f"maks. roznica {(h1 - ref).abs().max():.2e}")


sprawdz("CV-GRU: skladowe kandydata ograniczone do (-1,1)",
        n_t.real.abs().max() < 1 and n_t.imag.abs().max() < 1,
        f"maks. |Re| {n_t.real.abs().max():.4f}")

[OK   ] CV-GRU: bramka aktualizacji rzeczywista
[OK   ] CV-GRU: bramka resetu rzeczywista
[OK   ] CV-GRU: wartosci bramek w przedziale [1/2, 1)   min 0.5616, maks 0.9887
[OK   ] CV-GRU: stan ukryty zespolony
[OK   ] CV-GRU: ksztalt stanu   (5, 8)
[OK   ] CV-GRU: zgodnosc z rownaniem aktualizacji stanu   maks. roznica 0.00e+00
[OK   ] CV-GRU: skladowe kandydata ograniczone do (-1,1)   maks. |Re| 0.9917


True

## 9. Kontrola gradientów

Porównanie gradientów wyznaczonych analitycznie z oszacowaniem uzyskanym metodą
różnic skończonych, w arytmetyce podwójnej precyzji. Funkcja straty odnosi się do
losowego celu, dzięki czemu gradienty są niezerowe również dla parametrów
przesunięcia.

In [16]:
def kontrola_gradientu(model, wejscie, eps=1e-6, n_prob=12, nazwa=""):
    model = model.double()
    wejscie = wejscie.to(torch.complex128)
    params = [p for p in model.parameters() if p.requires_grad]

    torch.manual_seed(11)
    with torch.no_grad():
        proba = model(wejscie)
    cel_r = torch.randn_like(proba.real if proba.is_complex() else proba)
    cel_i = torch.randn_like(cel_r)

    def strata():
        out = model(wejscie)
        if out.is_complex():
            return ((out.real - cel_r) ** 2 + (out.imag - cel_i) ** 2).sum()
        return ((out - cel_r) ** 2).sum()

    model.zero_grad()
    strata().backward()

    rng = np.random.default_rng(3)
    bledy = []
    for _ in range(n_prob):
        p = params[rng.integers(len(params))]
        idx = tuple(rng.integers(s) for s in p.shape)
        analit = p.grad[idx].item()

        with torch.no_grad():
            org = p[idx].item()
            p[idx] = org + eps; l_plus = strata().item()
            p[idx] = org - eps; l_minus = strata().item()
            p[idx] = org

        numer = (l_plus - l_minus) / (2 * eps)
        mian = max(abs(analit), abs(numer), 1e-8)
        bledy.append(abs(analit - numer) / mian)

    maks = max(bledy)
    sprawdz(f"gradienty: {nazwa}", maks < 1e-4,
            f"maks. blad wzgledny {maks:.2e}")
    return maks


torch.manual_seed(1)
kontrola_gradientu(ComplexConv2d(2, 3, k=3, padding=1),
                   torch.complex(torch.randn(2, 2, 6, 6), torch.randn(2, 2, 6, 6)),
                   nazwa="warstwa splotowa")

kontrola_gradientu(ComplexBatchNorm2d(3).train(),
                   torch.complex(torch.randn(8, 3, 5, 5), torch.randn(8, 3, 5, 5)),
                   nazwa="normalizacja wsadowa")

kontrola_gradientu(ComplexConvBlock(2, 3, pool=(2, 2)).train(),
                   torch.complex(torch.randn(4, 2, 8, 8), torch.randn(4, 2, 8, 8)),
                   nazwa="pelny blok splotowy")

[OK   ] gradienty: warstwa splotowa   maks. blad wzgledny 1.04e-08
[OK   ] gradienty: normalizacja wsadowa   maks. blad wzgledny 4.71e-09
[OK   ] gradienty: pelny blok splotowy   maks. blad wzgledny 4.76e-09


4.758037064292779e-09

In [17]:

class OpakowanieGRU(nn.Module):
    def __init__(self, cin, hidden):
        super().__init__()
        self.cell = ComplexGRUCell(cin, hidden)
        self.hidden = hidden

    def forward(self, x):
        h = torch.zeros(x.shape[0], self.hidden, dtype=x.dtype, device=x.device)
        for t in range(x.shape[1]):
            h = self.cell(x[:, t], h)
        return h


torch.manual_seed(2)
kontrola_gradientu(OpakowanieGRU(6, 4),
                   torch.complex(torch.randn(3, 5, 6), torch.randn(3, 5, 6)),
                   nazwa="jednostka rekurencyjna")

[OK   ] gradienty: jednostka rekurencyjna   maks. blad wzgledny 4.15e-07


4.150721766620521e-07

## 10. Test integracyjny

Sprawdzenie przepływu przez pełny model, zgodności liczby parametrów ze
specyfikacją oraz dotarcia gradientu do wszystkich parametrów uczonych.

In [18]:
torch.manual_seed(0)
model = CVRCNN().to(DEV)
x = torch.complex(torch.randn(4, 1, N_BINS, N_FRAMES),
                  torch.randn(4, 1, N_BINS, N_FRAMES)).to(DEV)

out = model(x)
sprawdz("model: ksztalt wyjscia", tuple(out.shape) == (4, N_CLASSES),
        str(tuple(out.shape)))
sprawdz("model: wyjscie rzeczywiste", not out.is_complex())
sprawdz("model: brak wartosci nieskonczonych", torch.isfinite(out).all())

strata = F.cross_entropy(out, torch.randint(0, N_CLASSES, (4,), device=DEV))
strata.backward()

bez_grad = [n for n, p in model.named_parameters()
            if p.requires_grad and (p.grad is None or not torch.isfinite(p.grad).all())]
sprawdz("model: gradient dociera do wszystkich parametrow",
        len(bez_grad) == 0,
        "" if not bez_grad else f"brakujace: {bez_grad[:3]}")

n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
sprawdz("model: liczba parametrow zgodna ze specyfikacja",
        abs(n_par - 543819) < 10, f"{n_par:,}")

[OK   ] model: ksztalt wyjscia   (4, 6)
[OK   ] model: wyjscie rzeczywiste
[OK   ] model: brak wartosci nieskonczonych
[OK   ] model: gradient dociera do wszystkich parametrow
[OK   ] model: liczba parametrow zgodna ze specyfikacja   543,819


True

## 11. Podsumowanie

In [19]:
zdane = sum(1 for _, ok in wyniki if ok)
print("=" * 62)
print(f"WYNIK: {zdane} / {len(wyniki)} testow zakonczonych powodzeniem")
print("=" * 62)

niepowodzenia = [n for n, ok in wyniki if not ok]
if niepowodzenia:
    print("\nTesty zakonczone niepowodzeniem:")
    for n in niepowodzenia:
        print("  -", n)
else:
    print("\nWszystkie sprawdzane wlasnosci zostaly potwierdzone.")

WYNIK: 39 / 39 testow zakonczonych powodzeniem

Wszystkie sprawdzane wlasnosci zostaly potwierdzone.
